# 05 — Per-agent smoke (Backlog / Builder / Git / QA / UAT)

Each section has two parts:

1. **Capability boundary inspection** — read the agent's `allowed_tools` and prove it matches the spec (always runs, no LLM cost).
2. **Live smoke run** — gated on `HSB_NOTEBOOK_RUN_LIVE=1` + the relevant scratch fixtures.

Live runs cost tokens and require credentials. Default state: inspection only.

Existing fixtures in `src/fixture/` (`branch_test.py`, `pr_base_test.py`, `pr_title_test.py`) are minimal Builder targets you can reuse.

In [ ]:
import asyncio
import os
import re

from _helpers import assert_g1_safe, ensure_src_on_path, gated, live_mode

ROOT = ensure_src_on_path()

## Backlog Agent

**Allow-list spec:** `mcp__linear__create_issue`, `mcp__linear__list_issues`, `mcp__linear__get_issue`, `Read` only — must NOT include `update`, `delete`, `Bash`, `Edit`, `Write`.

**Idempotency:** the system prompt's IDEMPOTENCY RULE forces a `list_issues` pre-flight. We can't unit-test the LLM's compliance, but we can confirm the rule literal appears in the prompt.

In [ ]:
src = (ROOT / "src/hsb/agents/backlog_agent.py").read_text()
must_have = [
    "mcp__linear__create_issue",
    "mcp__linear__list_issues",
    "mcp__linear__get_issue",
]
must_not = ["mcp__linear__update_issue", "mcp__linear__delete", '"Edit"', '"Write"']
for tok in must_have:
    assert tok in src, f"Backlog allow-list missing {tok}"
for tok in must_not:
    # Search only inside ClaudeAgentOptions(...) by a simple proxy: anywhere in file.
    if tok in src:
        # If the token appears, make sure it's a forbidden-list mention, not an allow-list.
        # Simple heuristic: look for it inside an allowed_tools=[...] literal.
        for m in re.finditer(r"allowed_tools\s*=\s*\[(.*?)\]", src, re.DOTALL):
            assert tok not in m.group(1), f"Backlog allow-list contains forbidden {tok}"
assert "IDEMPOTENCY RULE" in src, "Backlog system prompt missing IDEMPOTENCY RULE"
print("Backlog: allow-list matches spec, IDEMPOTENCY RULE present in prompt")

In [ ]:
if not live_mode() or not os.environ.get("HSB_NOTEBOOK_PLAN_MD"):
    print(gated("Backlog live run (set HSB_NOTEBOOK_PLAN_MD=/path/to/plan.md)"))
else:
    assert_g1_safe()
    from hsb.agents.backlog_agent import run_backlog_agent
    from hsb.contracts.backlog import BacklogInput, ProjectContext

    inp = BacklogInput(
        plan_source=os.environ["HSB_NOTEBOOK_PLAN_MD"],
        project_context=ProjectContext(
            name="hsb-nb", repository="nb", technical_stack=["python"]
        ),
    )
    out = run_backlog_agent(inp)
    print(f"EPICs: {len(out.epics)} | first EPIC: {out.epics[0].title!r}")
    # IDEMPOTENCY: rerun should NOT create new EPICs (BKPK-05). The agent should
    # find the existing EPIC and reuse. This is a probabilistic check — examine
    # the Linear team after the second run to verify no duplicate EPIC titles.
    out2 = run_backlog_agent(inp)
    same = len(out2.epics) == len(out.epics)
    print("rerun returned", "same" if same else "different", "EPIC count")

## Builder Agent

**Capability boundary (BLDR-04):** `Read`, `Edit`, `Write` + `Bash` for `pytest|ruff|mypy|python` only. **No** Linear MCP, **no** git, **no** `gh`.

Schema-level guard: `BuilderOutput` rejects `git_branch` / `pr_url` / `linear_status` (covered in notebook 01).

In [ ]:
src = (ROOT / "src/hsb/agents/builder_agent.py").read_text()
# allowed_tools list literal
m = re.search(r"allowed_tools\s*=\s*\[(.*?)\]", src, re.DOTALL)
assert m, "Builder allowed_tools literal not found"
block = m.group(1)
for ok in ['"Read"', '"Edit"', '"Write"', "pytest", "ruff", "mypy"]:
    assert ok in block, f"Builder allow-list missing {ok}"
for forbidden in ["mcp__linear__", '"Agent"', '"git "', "gh pr "]:
    assert forbidden not in block, f"Builder allow-list contains forbidden {forbidden}"
# Top-level: NO mcp_servers configuration in Builder.
assert "mcp_servers=" not in src or "mcp_servers=None" in src, (
    "Builder ships an MCP server"
)
print("Builder: allow-list matches BLDR-04 (no Linear, no git, no Agent)")

In [ ]:
scratch = os.environ.get("HSB_NOTEBOOK_SCRATCH_DIR")
if not live_mode() or not scratch:
    print(gated("Builder live run (set HSB_NOTEBOOK_SCRATCH_DIR=/tmp/some-repo)"))
else:
    assert_g1_safe()
    from hsb.agents.builder_agent import run_builder_agent
    from hsb.contracts.builder import BuilderInput, RepositoryContext

    inp = BuilderInput(
        work_item_id="LIN-NB-1",
        issue_description="Add a one-line print to scratch.py",
        acceptance_criteria=['File scratch.py prints "hello"'],
        plan_source="plan.md",
        repository_context=RepositoryContext(
            root_path=scratch, technical_stack=["python"]
        ),
    )
    out = run_builder_agent(inp)
    print("status =", out.implementation_status)
    print("files =", [f.path for f in out.files_changed])

## Git Agent

**Allow-list:** `gh pr create/list/view/diff` (NOT `gh pr merge`), `git checkout/push/rebase/log/fetch/add/commit/status` (NOT `git merge`). All `gh pr list` calls must include `--limit 100` (Pitfall 4) and `git push` is `--force-with-lease` only (Pitfall 3).

In [ ]:
src = (ROOT / "src/hsb/agents/git_agent.py").read_text()
m = re.search(r"allowed_tools\s*=\s*\[(.*?)\]", src, re.DOTALL)
assert m, "Git allowed_tools literal not found"
block = m.group(1)
for forbidden in ["gh pr merge", "git merge", '"Edit"', '"Write"', "mcp__linear__"]:
    assert forbidden not in block, f"Git allow-list contains forbidden {forbidden}"
assert "--force-with-lease" in block, "Git allow-list missing --force-with-lease"
# Pitfall 4 — system prompt must mention --limit 100
assert "--limit 100" in src, "Git system prompt missing --limit 100 (Pitfall 4)"
print("Git: no merge tools, --force-with-lease only, --limit 100 in prompt")

## QA Agent

**Allow-list:** `Read`, `Bash(gh pr diff *)`, `Bash(gh pr view *)` only. **No** Edit/Write, **no** `gh pr create`, **no** Linear MCP. Cycle cap is enforced by the Pydantic `model_validator` on `QAOutput` (the deterministic last line of defense — see notebook 01).

In [ ]:
src = (ROOT / "src/hsb/agents/qa_agent.py").read_text()
m = re.search(r"allowed_tools\s*=\s*\[(.*?)\]", src, re.DOTALL)
block = m.group(1)
must_have = ['"Read"', "gh pr diff", "gh pr view"]
must_not = ["gh pr create", "gh pr merge", '"Edit"', '"Write"', "mcp__linear__"]
for tok in must_have:
    assert tok in block, f"QA allow-list missing {tok}"
for tok in must_not:
    assert tok not in block, f"QA allow-list contains forbidden {tok}"
# QAAG-04 system-prompt instruction — third defense layer (the hardest is the validator).
assert "CYCLE CAP" in src.upper(), "QA prompt missing cycle-cap instruction"
print("QA: 3-tool allow-list intact, CYCLE CAP instruction present")

## UAT Agent

**Allow-list:** `Read`, `Glob`, `Grep`, `Bash`. **No** Edit/Write/Agent. **`mcp_servers=None`** — UAT cannot write to Linear; the WIO/Global Orchestrator persist findings.

**Scope boundary:** every prompt prepends `SCOPE BOUNDARY:` and lists `[AC-N]` literals.

In [ ]:
src = (ROOT / "src/hsb/agents/uat_agent.py").read_text()
assert 'allowed_tools=["Read", "Glob", "Grep", "Bash"]' in src, (
    "UAT allow-list literal not found"
)
assert "mcp_servers=None" in src, "UAT must set mcp_servers=None"
assert "SCOPE BOUNDARY" in src, "UAT prompt missing SCOPE BOUNDARY"
# UAT must not import linear_agent (AI-SPEC §6 air-gap)
assert "linear_agent" not in src, "UAT module must not import linear_agent"
print(
    "UAT: 4-tool allow-list, mcp_servers=None, SCOPE BOUNDARY present, no linear_agent import"
)

In [ ]:
if not live_mode():
    print(gated("UAT live run"))
else:
    assert_g1_safe()
    from hsb.agents.uat_agent import run_uat_and_validate

    out = asyncio.run(
        run_uat_and_validate(
            user_story_id="US-NB-1",
            acceptance_criteria=["The README.md file exists at repo root"],
            uat_cycle=1,
        )
    )
    print("status =", out.overall_status, "| scenarios:", len(out.scenarios))